# Whisker-Pole Contact Classifier — Inference

Run the trained EfficientNet-B3 on a video and produce:
1. **Per-frame CSV** — every frame with its contact prediction (0/1) and probability
2. **Interval CSV** — contiguous contact regions in `Start,End` format

In [1]:
import sys, os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

INFERENCE_DIR = os.path.dirname(os.path.abspath("__file__"))
if INFERENCE_DIR not in sys.path:
    sys.path.insert(0, INFERENCE_DIR)

from utils import (
    load_model,
    get_inference_transforms,
    VideoFrameDataset,
    run_inference,
    frames_to_intervals,
)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch 2.7.1+cu126
CUDA available: True
GPU: NVIDIA GeForce GTX 1070 Ti


## 1 — Configuration

In [2]:
# ========================  PATHS  ========================

VIDEO_PATH = "/mnt/c/Users/wanglab/Desktop/Club Like Endings/102625_1/102625_1.mp4"


CHECKPOINT_PATH = os.path.join(
    os.path.dirname(INFERENCE_DIR),
    "Training", "checkpoints", "contact_classifier.pt"
)

# Output CSVs will be saved here
OUTPUT_DIR = os.path.join(INFERENCE_DIR, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================  SETTINGS  ========================

IMG_SIZE      = 256
BATCH_SIZE    = 16       # can be larger than training since no gradients
NUM_WORKERS   = 4
THRESHOLD     = 0.5      # probability threshold for contact
START_FRAME   = 0
END_FRAME     = 250_000  # only first 250k frames (second half has no pole)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

Device: cuda
Checkpoint: /home/wanglab/Deep-Learning/Contact Classification/Training/checkpoints/contact_classifier.pt
Output dir: /home/wanglab/Deep-Learning/Contact Classification/Inference/results


## 2 — Load Model

In [3]:
model = load_model(CHECKPOINT_PATH, DEVICE)

Loaded checkpoint from: /home/wanglab/Deep-Learning/Contact Classification/Training/checkpoints/contact_classifier.pt
  Config: {'img_size': 256, 'batch_size': 32, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'freeze_backbone': False, 'model_arch': 'efficientnet_b3'}
  Epoch: 2
  Val loss: 0.0090


## 3 — Create Video Dataset & DataLoader

In [4]:
transform = get_inference_transforms(IMG_SIZE)

dataset = VideoFrameDataset(
    video_path=VIDEO_PATH,
    start_frame=START_FRAME,
    end_frame=END_FRAME,
    transform=transform,
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Total frames to process: {len(dataset):,}")
print(f"Batches: {len(dataloader):,}")

VideoFrameDataset: frames 0–249999 (250,000 frames)
Total frames to process: 250,000
Batches: 15,625


## 4 — Run Inference

In [5]:
results_df = run_inference(model, dataloader, DEVICE, threshold=THRESHOLD)

n_contact = (results_df["contact"] == 1).sum()
n_no_contact = (results_df["contact"] == 0).sum()
print(f"\nTotal frames   : {len(results_df):,}")
print(f"Contact frames : {n_contact:,} ({100*n_contact/len(results_df):.1f}%)")
print(f"No contact     : {n_no_contact:,} ({100*n_no_contact/len(results_df):.1f}%)")
print()
results_df.head(10)

Inference: 100%|██████████| 15625/15625 [3:23:37<00:00,  1.28it/s]  



Total frames   : 250,000
Contact frames : 111,546 (44.6%)
No contact     : 138,454 (55.4%)



,frame,probability,contact
0,0,0.082138,0
1,1,0.086073,0
2,2,0.056738,0
3,3,0.098463,0
4,4,0.063964,0
5,5,0.062639,0
6,6,0.074656,0
7,7,0.062016,0
8,8,0.023138,0
9,9,0.032973,0


## 5 — Convert to Intervals & Save CSVs

In [6]:
# Per-frame CSV
per_frame_path = os.path.join(OUTPUT_DIR, "per_frame_predictions.csv")
results_df.to_csv(per_frame_path, index=False)
print(f"Saved per-frame predictions → {per_frame_path}")

# Contact intervals CSV
intervals_df = frames_to_intervals(results_df, label_col="contact", label_val=1)
intervals_path = os.path.join(OUTPUT_DIR, "contact_intervals.csv")
intervals_df.to_csv(intervals_path, index=False)
print(f"Saved contact intervals     → {intervals_path}")
print(f"\nFound {len(intervals_df)} contact intervals")
print()
intervals_df.head(20)

Saved per-frame predictions → /home/wanglab/Deep-Learning/Contact Classification/Inference/results/per_frame_predictions.csv
Saved contact intervals     → /home/wanglab/Deep-Learning/Contact Classification/Inference/results/contact_intervals.csv

Found 3487 contact intervals



,Start,End
0,262,262
1,267,269
2,477,477
3,1074,1074
4,1115,1115
5,1118,1118
6,1151,1153
7,1160,1161
8,1170,1170
9,1257,1257


## 6 — Quick Summary

In [7]:
# Interval length stats
intervals_df["length"] = intervals_df["End"] - intervals_df["Start"] + 1

print(f"Contact intervals: {len(intervals_df)}")
print(f"Total contact frames: {intervals_df['length'].sum():,}")
print(f"Avg interval length: {intervals_df['length'].mean():.1f} frames")
print(f"Min interval length: {intervals_df['length'].min()} frames")
print(f"Max interval length: {intervals_df['length'].max()} frames")
print(f"Median interval length: {intervals_df['length'].median():.0f} frames")

Contact intervals: 3487
Total contact frames: 111,546
Avg interval length: 32.0 frames
Min interval length: 1 frames
Max interval length: 2121 frames
Median interval length: 13 frames
